# Homework 2 — Problem 3: Prompt Engineering for Math Problem Solving

**NYU · CSCI-GA 3033-091: Intro to LLM Based Generative AI Systems · Fall 2026**  
**Instructors:** Parijat Dube and Chen Wang  
**TAs:** Raj Appana, Satyapragnya Kar  
**Due:** October 15, 2026 · **Points:** 25

**Name / NetID:** Mayank Garg / mg8948

**Objective:** Explore and refine different prompting techniques on one reproducibly selected GSM8K math word problem. This adapts the existing prompt-engineering exercise, retaining its six tasks and five function interfaces.

**Rubric:** Dataset setup **2** + model selection/justification **4** + prompting **8** + refinement **4** + evaluation **4** + summary **3** = **25 points**.

The supplied backend runs a configurable Hugging Face instruction model locally; the default is `google/flan-t5-base`. CPU is supported (generation can be slow); a GPU is optional. The first dataset/model load needs internet access and downloads public assets, but **no paid API or API key is required**. Complete the starter TODOs before enabling the experiment. Outputs are intentionally cleared; no model results are precomputed.


## Task 1: Dataset Setup (2 points)
Use Hugging Face `datasets` to load GSM8K and select **one test item with seed 42**. Run the supplied setup, record the selected test index, and keep that same question for all methods. Keep the reference answer separate from model inputs: it is used only for scoring after generation.

### Google Colab: upload just this notebook
1. Open [Google Colab](https://colab.research.google.com/) and choose **File → Upload notebook**. No companion files, repository access, or API keys are needed.
2. Use **Runtime → Change runtime type** to request a GPU if available; CPU also works for FLAN-T5-base. Run cells in order.
3. The next cell installs only missing/incompatible problem-specific packages and preserves the installed Torch stack. If it installs packages, **restart the session** and run from the top before importing models.
4. Complete the TODOs, then enable `RUN_EXPERIMENT`. Download your executed notebook with **File → Download → Download .ipynb**; Colab runtime files do not persist indefinitely.

This also works in a local Jupyter kernel with a compatible PyTorch installation. No author-specific environment or `requirements.txt` is required.

In [1]:
# Self-contained setup: preserve the preinstalled Torch/vision/audio stack.
import importlib.metadata as metadata
from pathlib import Path
import subprocess
import sys
import tempfile
from packaging.requirements import Requirement

P3_PACKAGES = ("transformers>=4.57,<5", "datasets>=3,<5",
               "sentencepiece>=0.2,<0.3", "pandas>=2.1,<4")
try:
    torch_version = metadata.version("torch")
except metadata.PackageNotFoundError:
    raise RuntimeError("Use a Colab runtime with PyTorch, or install PyTorch in your Jupyter environment first.")
if not Requirement("torch>=2.4,<3").specifier.contains(torch_version):
    raise RuntimeError("This exercise requires compatible PyTorch 2.4+. Use a fresh supported runtime; setup will not replace your GPU stack.")
missing = []
for spec in P3_PACKAGES:
    requirement = Requirement(spec)
    try:
        installed = metadata.version(requirement.name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not requirement.specifier.contains(installed):
        missing.append(spec)
if missing:
    constraints = []
    for name in ("torch", "torchvision", "torchaudio", "numpy"):
        try:
            constraints.append(f"{name}=={metadata.version(name)}")
        except metadata.PackageNotFoundError:
            pass
    with tempfile.TemporaryDirectory(prefix="hw2-p3-install-") as directory:
        constraint_file = Path(directory) / "constraints.txt"
        constraint_file.write_text("\n".join(constraints) + "\n")
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade-strategy",
                        "only-if-needed", "-c", str(constraint_file), *missing], check=True)
    raise RuntimeError("Packages installed. Restart the Colab session/Jupyter kernel, then run from the top.")
print({"python": sys.version.split()[0], "torch": torch_version,
       **{Requirement(spec).name: metadata.version(Requirement(spec).name) for spec in P3_PACKAGES}})

{'python': '3.13.15', 'torch': '2.11.0+cu130', 'transformers': '4.57.6', 'datasets': '4.8.5', 'sentencepiece': '0.2.2', 'pandas': '2.2.3'}


In [2]:
import datasets
import random

# The reference solution stays outside all prompting functions.
dataset = datasets.load_dataset("gsm8k", "main")
test_set = dataset["test"]
random_index = random.Random(42).randint(0, len(test_set) - 1)
selected_problem = test_set[random_index]
question = selected_problem["question"]
reference_answer = selected_problem["answer"]
print(f"Selected test index: {random_index}")
print(f"Selected question: {question}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Selected test index: 1309
Selected question: The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, who raises $430, and Maryam raises $400 more than Sarah, who raises $300. How much money, in dollars, did they all raise in total?


## Task 2: Model Selection (4 points)
Choose and justify a suitable instruction-following model. The supplied **ungraded** local backend defaults to `google/flan-t5-base` using `AutoTokenizer` and `AutoModelForSeq2SeqLM`. You may change `MODEL_NAME` to a compatible encoder–decoder model; causal models require a different backend. Explain instruction tuning, resource requirements, context/output budgets, and limitations for multi-step arithmetic. A runnable default is not a model-selection justification.

Model weights are loaded only when you call `load_backend()` (Task 5 does so when enabled). Greedy decoding is held fixed for a fair prompt comparison. Oversize input prompts raise an error rather than silently dropping exemplars or the question. If this happens, shorten the prompts consistently or select a model with a genuinely larger input budget; do not simply increase the limit past the model's documented support.

**Model justification — `google/flan-t5-large` (780M parameters, encoder–decoder).**

- **Why the FLAN-T5 family.** The supplied backend uses `AutoModelForSeq2SeqLM`, so only encoder–decoder models are compatible. Within that constraint, FLAN-T5 is the standard *instruction-tuned* choice.
  - It is T5 fine-tuned on the FLAN collection of ~1.8K tasks phrased as natural-language instructions (Chung et al., 2022, "Scaling Instruction-Finetuned Language Models").
  - That mixture includes **chain-of-thought exemplars**, so the model has seen both "answer only" and "reason, then answer" formats. Few-shot numeric and CoT prompts are therefore meaningful for it, unlike for the plain pre-trained T5.
  - *Caveat:* the FLAN CoT mixture draws on GSM8K's **training** split. The test item used here is held out, but the model is familiar with the problem style.
- **Why `large` instead of the default `base`.**
  - Multi-step arithmetic is where small models fail first. `base` (250M) rarely produces a correct multi-step chain, and the FLAN-T5 paper reports reasoning ability improving markedly with scale.
  - `large` (780M) is the biggest size that is still comfortable everywhere: about 3.1 GB of fp32 weights, which runs on CPU in a few seconds per answer or easily on a Colab T4.
  - `xl` (3B, ~11 GB fp32) would likely reason better, but it needs a GPU and is slow on CPU. `xxl` (11B) does not fit a free T4.
  - The choice can be changed through `MODEL_NAME` without touching the backend.
- **Context and output budgets.**
  - The T5 tokenizer advertises a **512-token input limit**, and the backend errors out instead of truncating. Measured input sizes for the selected question are 122 / 152 / 214 / 305 tokens (one-shot, two-shot, CoT, refined), all well under the 512-token limit.
  - `MAX_NEW_TOKENS = 128` is enough for a short CoT and a final answer line. A cut-off output triggers the backend's "no EOS" warning.
  - Decoding is greedy (`do_sample=False`) for every method, so differences between methods come from the prompt alone.
- **Limitations for multi-step arithmetic.**
  1. A 780M model computes digits by pattern-matching tokens, so carries, multiplications and percentages are error-prone. It has no calculator.
  2. Errors compound across reasoning steps, which is exactly what GSM8K tests.
  3. **T5's SentencePiece vocabulary has no newline token.** Line breaks in prompts are seen as spaces, and outputs never contain `\n`. The supplied scorer only accepts an `Answer:` marker at the start of the *last line*, so a single-line CoT output would be marked unparseable even when its answer is right. My reasoning prompts therefore post-process the output: they insert one line break before the model's own final `Answer:` marker. The number itself is never changed or inferred, and the raw model output is printed alongside for transparency.


In [3]:
# Supplied infrastructure; no model download occurs when this cell is defined.
import os
import warnings

MODEL_NAME = os.environ.get("PROMPT_MODEL_NAME", "google/flan-t5-large")
DEVICE = os.environ.get("PROMPT_DEVICE", "auto")  # auto -> CUDA if available, else CPU
MAX_NEW_TOKENS = 128
backend = None


def load_backend(model_name=MODEL_NAME, device=DEVICE, max_new_tokens=MAX_NEW_TOKENS,
                 input_max_tokens=None):
    """Return a prompt -> text callable. Loading/generation errors propagate.

    input_max_tokens may LOWER an advertised limit. If a custom model advertises
    no finite limit, pass a limit justified by that model's documentation.
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    if not isinstance(max_new_tokens, int) or isinstance(max_new_tokens, bool) or max_new_tokens <= 0:
        raise ValueError("max_new_tokens must be a positive integer")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    limits = []
    for value in (getattr(tokenizer, "model_max_length", None),
                  getattr(model.config, "max_position_embeddings", None),
                  getattr(model.config, "n_positions", None)):
        # Tokenizers sometimes use a huge sentinel to mean "unknown".
        if isinstance(value, int) and 0 < value < 1_000_000:
            limits.append(value)
    if input_max_tokens is not None:
        if not isinstance(input_max_tokens, int) or isinstance(input_max_tokens, bool) or input_max_tokens <= 0:
            raise ValueError("input_max_tokens must be a positive integer")
        limits.append(input_max_tokens)
    if not limits:
        raise ValueError("No finite input limit advertised; provide a documented input_max_tokens.")
    input_limit = min(limits)

    def generate_text(full_prompt):
        if not isinstance(full_prompt, str) or not full_prompt.strip():
            raise ValueError("full_prompt must be nonempty text")
        inputs = tokenizer(full_prompt, return_tensors="pt", truncation=False)
        token_count = inputs["input_ids"].shape[-1]
        if token_count > input_limit:
            raise ValueError(f"Prompt has {token_count} tokens; limit is {input_limit}. No truncation was applied.")
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        eos = model.generation_config.eos_token_id
        eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
        if generated[0, -1].item() not in eos_ids:
            warnings.warn("Generation ended without EOS; output may be cut off by the token budget.", RuntimeWarning)
        return tokenizer.decode(generated[0], skip_special_tokens=True).strip()

    return generate_text


## Task 3: Prompt Engineering (8 points)
Implement the four original interfaces below (**2 points each**). The supplied backend is a callable `backend(full_prompt)` returning generated text.

1. `generate_solution(prompt, problem)`: combine the instruction/examples with **only the question string**, then call the backend. Do not catch backend errors and turn them into answer strings.
2. `one_shot_prompting_numeric(problem_to_solve)`: use one worked-example **answer containing only a number**, with no reasoning in that example's answer.
3. `two_shot_prompting_numeric(problem_to_solve)`: use two such numeric-only example answers.
4. `two_shot_cot_prompting(problem_to_solve)`: use two examples with worked arithmetic reasoning and a final `Answer: <number>` line.

Use the same simple apples/train examples across numeric and CoT variants so the main change is the answer format. For example, you may ask about buying additional apples or distance at a constant train speed; supply your own correct answers. Require numeric-only model output for numeric prompts, and an explicit final `Answer: <number>` line for reasoning prompts. Never insert `reference_answer`, the GSM8K answer rationale, or the selected item dictionary into a prompt.


In [4]:
import re

# Shared exemplars (apples / train), identical across the numeric and CoT variants, so the main
# difference between methods is the *answer format*. These are my own problems with checked answers.
EX_APPLES_Q = ("Tom has 3 apples. He buys 4 bags of apples, and each bag has 5 apples. "
               "How many apples does Tom have now?")
EX_APPLES_NUM = "23"
EX_APPLES_COT = ("Tom starts with 3 apples. The bags hold 4 * 5 = 20 apples. "
                 "In total he has 3 + 20 = 23 apples. Answer: 23")

EX_TRAIN_Q = ("A train travels at a constant speed of 60 miles per hour. "
              "How many miles does it travel in 2.5 hours?")
EX_TRAIN_NUM = "150"
EX_TRAIN_COT = ("Distance equals speed times time. The train travels 60 * 2.5 = 150 miles. "
                "Answer: 150")

NUMERIC_INSTRUCTION = ("Solve the math word problem. Reply with only the final answer as a number, "
                       "with no words, units or explanation.")
COT_INSTRUCTION = ("Solve the math word problem. Reason step by step, showing each calculation, "
                   "then end with the final answer in the form 'Answer: <number>'.")


def _example(question, answer):
    return f"Q: {question}\nA: {answer}"


def generate_solution(prompt, problem):
    """Combine prompt and question, call backend, and return text (2 points).

    `prompt` holds the instruction and exemplars; `problem` must be ONLY the question string.
    The question is appended exactly once as the final 'Q:' and the model continues after 'A:'.
    Backend errors (missing model, over-length prompt) are not caught: they propagate.
    """
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be nonempty text")
    if not isinstance(problem, str) or not problem.strip():
        raise ValueError("problem must be the question text only")
    if "####" in problem:                      # guard: never pass a GSM8K answer/rationale in
        raise ValueError("problem appears to contain a GSM8K answer; pass the question only")
    full_prompt = f"{prompt.strip()}\n\nQ: {problem.strip()}\nA:"
    if backend is None:
        raise RuntimeError("backend is not loaded; call load_backend() first")
    return backend(full_prompt)


def _answer_on_own_line(text):
    """FLAN-T5 cannot emit '\\n' (its vocabulary has no newline token), so a reasoning answer comes
    back on one line. Insert a single line break before the model's LAST 'Answer:'/'Final answer:'
    marker so the supplied strict scorer can find it. Nothing else is changed; if the model wrote no
    marker, the text is returned unchanged (and will be scored as unparseable)."""
    matches = list(re.finditer(r"(final\s+answer|answer)\s*:", text, flags=re.IGNORECASE))
    if not matches:
        return text
    i = matches[-1].start()
    return text[:i].rstrip() + "\n" + text[i:].strip()


def one_shot_prompting_numeric(problem_to_solve):
    """One exemplar whose answer is numeric only (2 points)."""
    prompt = f"{NUMERIC_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_NUM)}"
    return generate_solution(prompt, problem_to_solve)


def two_shot_prompting_numeric(problem_to_solve):
    """Two exemplars whose answers are numeric only (2 points)."""
    prompt = (f"{NUMERIC_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_NUM)}\n\n"
              f"{_example(EX_TRAIN_Q, EX_TRAIN_NUM)}")
    return generate_solution(prompt, problem_to_solve)


def two_shot_cot_prompting(problem_to_solve):
    """Two worked-reasoning exemplars with explicit final answers (2 points)."""
    prompt = (f"{COT_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_COT)}\n\n"
              f"{_example(EX_TRAIN_Q, EX_TRAIN_COT)}")
    raw = generate_solution(prompt, problem_to_solve)
    RAW_OUTPUTS["Two-shot CoT"] = raw
    return _answer_on_own_line(raw)


RAW_OUTPUTS = {}   # untouched model text for the reasoning methods (shown next to the scored output)

# Quick offline check of prompt construction (no model needed): show the exact CoT prompt text.
print(f"{COT_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_COT)}\n\n{_example(EX_TRAIN_Q, EX_TRAIN_COT)}"
      f"\n\nQ: <selected GSM8K question>\nA:")

Solve the math word problem. Reason step by step, showing each calculation, then end with the final answer in the form 'Answer: <number>'.

Q: Tom has 3 apples. He buys 4 bags of apples, and each bag has 5 apples. How many apples does Tom have now?
A: Tom starts with 3 apples. The bags hold 4 * 5 = 20 apples. In total he has 3 + 20 = 23 apples. Answer: 23

Q: A train travels at a constant speed of 60 miles per hour. How many miles does it travel in 2.5 hours?
A: Distance equals speed times time. The train travels 60 * 2.5 = 150 miles. Answer: 150

Q: <selected GSM8K question>
A:


## Task 4 & 5: Prompt Refinement & Evaluation (4 + 4 points)

### Task 4 — Refinement (4 points)
Implement the original `refined_prompting` interface. Try a deliberate variation (for example, ask the model to check its arithmetic and units), and explain what changed and why. Keep a parseable final answer and do not reveal the reference answer. Compare the chosen refinement with the original three methods; do not select a prompt by copying the held-out answer.

### Task 5 — Evaluation (4 points)
The supplied helper records each method's raw output, extracted final number, exact numeric match, and wall-clock generation time. **Set `RUN_EXPERIMENT = True`** after completing your functions and running Tasks 1–3. Record the model/configuration and inspect the measured table; discuss correctness, format compliance, reasoning quality, and the refinement's observed effect. Latency excludes model loading but includes tokenization/generation/decoding; the first generation can include warm-up overhead.

Scoring compares a numeric-only response or an explicit final `Answer:`, `Final answer:`, or `####` line against the number after GSM8K's last `####`. It supports signed numbers, valid thousands commas, and decimals using exact `Decimal` values, not substring matching. An absent/malformed prediction is **incorrect**, never a match to another missing value; a malformed reference raises an error. This intentionally strict format does not infer an answer from intermediate calculations, fractions, units, or arbitrary prose. Inspect raw output to distinguish reasoning errors from format errors. Backend exceptions propagate and stop the run rather than being counted as mathematical answers.

One question gives a per-method binary result, **not** a reliable benchmark accuracy. If you refine after seeing its result, explain this test-item tuning and avoid generalization claims.

**TODO — Refinement rationale and evaluation discussion:**

**Refinement rationale.** `refined_prompting` keeps the same two exemplars (apples, train) as the two-shot CoT prompt, so the comparison isolates the effect of *structure*. It changes four things, each aimed at a typical GSM8K failure:

1. **"Given:" list first.** The model restates the quantities in the problem before computing, so no number in the question is silently dropped.
2. **Numbered equations** (`Step k: … = result`). There is one operation per step, so each arithmetic operation is short and local; long sentences mix several operations and invite carry errors.
3. **Explicit check.** The model verifies units and that the last step answers *what is asked*. This targets the common error of stopping at an intermediate quantity, e.g. reporting a per-day value when the question asks for a weekly total.
4. **Same strict final format**, `Answer: <number>`, with a plain number and no units.

The refinement was designed before seeing the selected question's outputs and was not tuned to this item. The reference answer is never shown to any prompt.

**Evaluation discussion.**

**Evaluation discussion** *(measured run: `google/flan-t5-large` on a Colab GPU, greedy decoding, `max_new_tokens=128`; GSM8K test index 1309, seed 42; reference number 2280)*

| Method | Scored output | Prediction | Format valid | Correct | Time |
|---|---|---|---|---|---|
| One-shot numeric | `900` | 900 | ✓ | ✗ | 0.14 s |
| Two-shot numeric | `900` | 900 | ✓ | ✗ | 0.14 s |
| Two-shot CoT | reasoning text with no final `Answer:` line | — | ✗ | ✗ | 2.14 s |
| Refined | `Answer: 630` | 630 | ✓ | ✗ | 0.21 s |

- **Correctness.** No method matches the reference.
  - *The reference appears to be mislabeled.* From the question: Kim = 430 + 320 = 750, Alexandra = 430, Maryam = 300 + 400 = 700, Sarah = 300, so the total is **2180**. GSM8K's rationale instead sums 750 + 430 + **400** + 700 = 2280, using Maryam's $400 *difference* in place of Sarah's $300.
  - I kept the supplied strict scoring against the `####` value unchanged. A correct answer of 2180 would therefore also have been marked wrong, but no method produced 2180, so the conclusion that all four methods failed holds against either value.
- **Format compliance.**
  - Both numeric prompts complied exactly: a bare number.
  - The refined prompt produced a parseable `Answer: 630`, but it **ignored the requested structure** (no "Given:" list, steps or check). It output only the answer line.
  - Two-shot CoT was **unparseable**. The model began with `Answer:` and then wrote its reasoning, ending "The total is 820 + 920 = $1440." with no closing `Answer:` marker. The line-break helper only moves the model's *last* marker, which here was the opening one, so the final line remained reasoning text and the strict scorer rejected it. This is a format failure, on top of the reasoning errors.
- **Reasoning quality (CoT raw text).** The visible chain makes the errors inspectable. Almost every step is wrong:
  - 430 + 320 = 720, which should be 750;
  - 400 + 300 = 600, which should be 700;
  - Kim's $320 is added a second time (720 + 320 = 820);
  - a number not in the problem, 920, appears;
  - 820 + 920 = 1440 is itself miscomputed (it is 1740);
  - Alexandra's and Sarah's amounts are never included.

  This fits the expected limits of a 780M model doing digit arithmetic without a calculator. The numeric answers (900, 630) have no rationale, so their source cannot be traced.
- **Effect of the refinement.**
  - Compared with CoT, it restored format compliance, going from unparseable to `Answer: 630`. It changed the prediction relative to the numeric prompts (900 → 630).
  - It did **not** improve correctness. Because the model skipped the Given/Step/Check structure entirely, the mechanism the refinement targets was never exercised.
  - The refinement was written before seeing this item's outputs and was not tuned afterwards.
- **Latency.** The numeric prompts (~0.14 s) and the refined prompt (0.21 s) have very short outputs. CoT took 2.14 s, about 15× longer, because it generated a long rationale. Latency is driven by *output* length, not prompt length (CoT 214 tokens vs. refined 305). Model loading is excluded, and the first call (one-shot) may include warm-up.
- **Limits of this test.** This is one question, giving a single pass/fail per method. Combined with a doubtful reference label, it cannot rank prompting methods; it only shows their failure modes. A fair comparison would need many GSM8K questions and probably a larger model (e.g. FLAN-T5-XL).


In [7]:
REFINED_INSTRUCTION = (
    "Solve the math word problem carefully. First list the quantities given in the problem. "
    "Then compute one step at a time, writing every calculation as an equation with its result. "
    "Check that each step uses the right numbers and units, and that the last step answers exactly "
    "what the question asks. End with 'Answer: <number>' using a plain number without units.")
EX_APPLES_REFINED = ("Given: 3 apples, 4 bags, 5 apples per bag. Step 1: apples in bags = 4 * 5 = 20. "
                     "Step 2: total apples = 3 + 20 = 23. Check: the question asks for all apples Tom "
                     "has now, which is 23. Answer: 23")
EX_TRAIN_REFINED = ("Given: speed 60 miles per hour, time 2.5 hours. Step 1: distance = speed * time "
                    "= 60 * 2.5 = 150. Check: the units are miles, as asked. Answer: 150")
def refined_prompting(problem_to_solve):
    """Implement a justified prompt refinement (4 points)."""
    # TODO: Build a refined prompt with an explicit final answer format.
    prompt = (f"{REFINED_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_REFINED)}\n\n"
              f"{_example(EX_TRAIN_Q, EX_TRAIN_REFINED)}")
    raw = generate_solution(prompt, problem_to_solve)
    RAW_OUTPUTS["Refined"] = raw
    return _answer_on_own_line(raw)


In [8]:
# Supplied evaluation helpers; defining this cell needs no model or dataset.
import re
from decimal import Decimal
from time import perf_counter

_NUMBER = re.compile(r"[+-]?(?:(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|\.\d+)", re.ASCII)
_MARKER = re.compile(r"^\s*(?:final\s+answer\s*:|answer\s*:|####)\s*(.*)$", re.IGNORECASE)


def parse_number(text):
    """Parse one finite decimal literal; reject malformed commas and extra text."""
    if not isinstance(text, str):
        return None
    text = text.strip().replace("−", "-")
    if not _NUMBER.fullmatch(text):
        return None
    return Decimal(text.replace(",", ""))


def extract_final_number(output):
    """Accept a numeric-only output or a final nonempty line with an answer marker."""
    if not isinstance(output, str) or not output.strip():
        return None
    direct = parse_number(output)
    if direct is not None:
        return direct
    final_line = output.strip().splitlines()[-1]
    match = _MARKER.fullmatch(final_line)
    return parse_number(match.group(1)) if match else None


def extract_reference_number(answer):
    """Require GSM8K's explicit #### delimiter; bad references are data errors."""
    if not isinstance(answer, str) or "####" not in answer:
        raise ValueError("Reference answer must contain GSM8K's #### delimiter")
    value = parse_number(answer.rsplit("####", 1)[1])
    if value is None:
        raise ValueError("Reference has a missing or malformed final number")
    return value


def evaluate_methods(problem, reference, methods):
    """Run method(question) only. Model exceptions propagate, separate from scoring."""
    expected = extract_reference_number(reference)
    rows = []
    for name, method in methods.items():
        start = perf_counter()
        output = method(problem)  # Never pass the reference to the prompting method.
        elapsed = perf_counter() - start
        prediction = extract_final_number(output)
        rows.append({"method": name, "raw_output": output,
                     "prediction": prediction, "reference": expected,
                     "format_valid": prediction is not None,
                     "correct": prediction is not None and prediction == expected,
                     "seconds": elapsed})
    return rows


def measured_results_table(rows):
    """Return a table of actual runs only; no synthetic/example outcomes."""
    import pandas as pd
    columns = ["method", "raw_output", "prediction", "reference", "format_valid", "correct", "seconds"]
    return pd.DataFrame(rows, columns=columns)


def summarize_results(rows):
    """Give a small results-dependent observation, not the student's written analysis."""
    if not rows:
        return "No methods have been evaluated; no performance conclusion is available."
    matching = [row["method"] for row in rows if row["correct"]]
    malformed = [row["method"] for row in rows if not row["format_valid"]]
    return (f"On this one question, final-number matches: {', '.join(matching) or 'none'}. "
            f"Unparseable outputs: {', '.join(malformed) or 'none'}. "
            "These observations do not establish a generally superior prompting method.")


RUN_EXPERIMENT = True  # Enable only after implementing TODOs and running dataset setup.
results = []
if RUN_EXPERIMENT:
    if backend is None:
        backend = load_backend()
    methods = {
        "One-shot numeric": one_shot_prompting_numeric,
        "Two-shot numeric": two_shot_prompting_numeric,
        "Two-shot CoT": two_shot_cot_prompting,
        "Refined": refined_prompting,
    }
    print(f"Model: {MODEL_NAME}; requested device: {DEVICE}; greedy decoding; max_new_tokens={MAX_NEW_TOKENS}")
    print(f"GSM8K test index: {random_index}; seed: 42\nQuestion: {question}")
    results = evaluate_methods(question, reference_answer, methods)
    from IPython.display import display
    display(measured_results_table(results))
print(summarize_results(results))


Model: google/flan-t5-large; requested device: auto; greedy decoding; max_new_tokens=128
GSM8K test index: 1309; seed: 42
Question: The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, who raises $430, and Maryam raises $400 more than Sarah, who raises $300. How much money, in dollars, did they all raise in total?


,method,raw_output,prediction,reference,format_valid,correct,seconds
0,One-shot numeric,900,900,2280,True,False,0.142084
1,Two-shot numeric,900,900,2280,True,False,0.139518
2,Two-shot CoT,\nAnswer: 430 + 320 = $720. Maryam raises $400...,None,2280,False,False,2.143818
3,Refined,\nAnswer: 630,630,2280,True,False,0.214420


On this one question, final-number matches: none. Unparseable outputs: Two-shot CoT. These observations do not establish a generally superior prompting method.


In [9]:
# Full outputs (the table truncates long text) + untouched model text for the reasoning methods
for row in results:
    print(f"=== {row['method']}  |  prediction={row['prediction']}  correct={row['correct']}  "
          f"time={row['seconds']:.2f}s")
    print("Scored output:", row["raw_output"])
    if row["method"] in RAW_OUTPUTS:
        print("Raw model text (before the line-break fix):", RAW_OUTPUTS[row["method"]])
    print()
print("Reference answer (GSM8K, shown only after generation):")
print(reference_answer)

# Prompt sizes vs. the 512-token input limit
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
_prompts = {
    "One-shot numeric": f"{NUMERIC_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_NUM)}",
    "Two-shot numeric": f"{NUMERIC_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_NUM)}\n\n{_example(EX_TRAIN_Q, EX_TRAIN_NUM)}",
    "Two-shot CoT": f"{COT_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_COT)}\n\n{_example(EX_TRAIN_Q, EX_TRAIN_COT)}",
    "Refined": f"{REFINED_INSTRUCTION}\n\n{_example(EX_APPLES_Q, EX_APPLES_REFINED)}\n\n{_example(EX_TRAIN_Q, EX_TRAIN_REFINED)}",
}
for name, p in _prompts.items():
    full = f"{p}\n\nQ: {question}\nA:"
    print(f"{name:<18} input tokens: {len(_tok(full)['input_ids'])} / 512")

=== One-shot numeric  |  prediction=900  correct=False  time=0.14s
Scored output: 900

=== Two-shot numeric  |  prediction=900  correct=False  time=0.14s
Scored output: 900

=== Two-shot CoT  |  prediction=None  correct=False  time=2.14s
Scored output: 
Answer: 430 + 320 = $720. Maryam raises $400 more than Sarah, who raises $300, so she raises $400 + 300 = $600. Kim raises $720 + $320 = $820. The total is 820 + 920 = $1440.
Raw model text (before the line-break fix): Answer: 430 + 320 = $720. Maryam raises $400 more than Sarah, who raises $300, so she raises $400 + 300 = $600. Kim raises $720 + $320 = $820. The total is 820 + 920 = $1440.

=== Refined  |  prediction=630  correct=False  time=0.21s
Scored output: 
Answer: 630
Raw model text (before the line-break fix): Answer: 630

Reference answer (GSM8K, shown only after generation):
Kim raises 320+430=<<320+430=750>>750 dollars.
Maryam raises 400+300=<<400+300=700>>700 dollars.
They raise 750+430+400+700=<<750+430+400+700=2280>>2280 

## Task 6: Summary (3 points)
Write **no more than 200 words** based on your **actual measured results**. Include the selected question/index and model, the three original prompting strategies, the refinement and its motivation, observed correctness/format/reasoning differences, resource or length issues, and why a single question cannot establish a winner. Cite evidence from the table and raw outputs rather than assuming CoT helps. If a run failed, describe the failure instead of inventing a result.

**TODO — Your summary:**

**Summary (193 words).** I evaluated `google/flan-t5-large` (780M-parameter instruction-tuned encoder–decoder; greedy decoding, 128 new tokens, Colab GPU) on GSM8K test item 1309 (seed 42), which asks for the total raised by four girls. One-shot and two-shot numeric prompts both answered 900. Two-shot CoT produced a visible chain, but almost every step was wrong (430 + 320 = 720; 820 + 920 = 1440), and it put "Answer:" at the start instead of the end, so the strict scorer found no final number. My refinement (list the givens, numbered equations, self-check, same exemplars) restored a parseable answer, but the model skipped the steps and replied "Answer: 630". No method matched. The reference itself appears mislabeled: the question implies 750 + 430 + 700 + 300 = 2180, but GSM8K's rationale adds the $400 difference instead of Sarah's $300, giving 2280. Lessons: this small model follows format instructions more reliably than reasoning instructions. CoT cost about 15× more latency (2.14 s vs 0.14 s). T5's tokenizer cannot emit newlines, which clashes with line-based answer parsing. All prompts fit within 512 tokens (maximum 305). One question, a doubtful label and a 780M model reveal failure modes, not a winning method.
